# 🇺🇬 Luganda Language Model Fine-tuning with LoRA

This notebook fine-tunes Gemma-2B on Luganda language curriculum data using Parameter-Efficient Fine-Tuning (LoRA).

**Expected Runtime:** 30-40 minutes on Google Colab's free T4 GPU

**What this does:**
- Trains a custom Luganda language tutor based on your curriculum
- Uses only ~10M trainable parameters (0.5% of the model)
- Saves the fine-tuned model for download and local deployment

---

## 🔗 VS Code Connection

**If running from VS Code:**
1. Open Command Palette (Ctrl+Shift+P / Cmd+Shift+P)
2. Select: **"Jupyter: Specify Jupyter Server for Connections"**
3. Choose: **"Existing: Specify the URI of an existing server"**
4. Enter: Your Colab runtime URL (see below)

**To get Colab URL:**
- In Colab: Click "Connect" → "Connect to a local runtime"
- Or use: https://colab.research.google.com/

**Alternatively:** Just upload this notebook to Colab directly!

---

## 📦 Step 1: Install Dependencies

Install all required libraries for training.

In [ ]:
!pip install -q transformers
!pip install -q torch
!pip install -q peft
!pip install -q accelerate
!pip install -q bitsandbytes
!pip install -q datasets
!pip install -q trl

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [615 lines of output]
      Checking for Rust toolchain....
      Rust not found, installing into a temporary directory
      Python reports SOABI: cp313-win_amd64
      Computed rustc target triple: x86_64-pc-windows-msvc
      Installation directory: C:\Users\DELL\AppData\Local\puccinialin\puccinialin\Cache
      
      Installing rust to C:\Users\DELL\AppData\Local\puccinialin\puccinialin\Cache\rustup
      warn: installing msvc toolchain without its prerequisites
      info: profile set to 'minimal'
      info: default host triple is x86_64-pc-windows-msvc
      info: syncing channel updates for 'stable-x86_64-pc-windows-msvc'
      info: latest update on 2025-12-11, rust version 1.92.0 (ded5c06cf 2025-12-08)
      info: downloading component 'cargo'
      info: downloading component 'rust-std'
      info: downloading component 'rustc'
      info: retryi

## 📤 Step 2: Upload Training Data

Upload your `luganda_training_data.json` file from:
`backend/data/training/luganda_training_data.json`

Click the folder icon on the left sidebar → Upload button

In [ ]:
import os
import json

# Verify the file was uploaded
if os.path.exists('luganda_training_data.json'):
    with open('luganda_training_data.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"✅ Training data loaded: {len(data)} examples")
    print(f"\nSample example:")
    print(json.dumps(data[0], indent=2, ensure_ascii=False)[:500] + "...")
else:
    print("❌ Please upload luganda_training_data.json using the file upload button")

## 🚀 Step 3: Fine-tune the Model

This cell will:
1. Load the Gemma-2B base model in 4-bit quantization
2. Configure LoRA for efficient training
3. Train on your Luganda curriculum data
4. Save checkpoints every 50 steps

**⏱️ Expected time: 30-40 minutes**

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import json

print("🔧 Configuration")
print(f"   GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# Load training data
print("\n📚 Loading training data...")
with open('luganda_training_data.json', 'r', encoding='utf-8') as f:
    training_data = json.load(f)

# Format data for training
def format_prompt(example):
    if example.get('input'):
        return f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    else:
        return f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}"""

formatted_data = [{'text': format_prompt(ex)} for ex in training_data]
dataset = Dataset.from_list(formatted_data)

print(f"✅ Loaded {len(dataset)} training examples")

# Model configuration
MODEL_NAME = "google/gemma-2b-it"

print("\n🤖 Loading base model with 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✅ Base model loaded")

# Prepare model for training
print("\n🔧 Configuring LoRA...")
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"✅ LoRA configured")
print(f"   Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"   Total parameters: {total_params:,}")

# Tokenize dataset
print("\n📝 Tokenizing dataset...")
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names
)
print("✅ Dataset tokenized")

# Training configuration
print("\n⚙️ Training configuration...")
training_args = TrainingArguments(
    output_dir="./luganda-gemma-2b-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=3,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

print("✅ Trainer initialized")
print("\n🎯 Starting training...")
print("   This will take approximately 30-40 minutes on T4 GPU\n")

# Start training
trainer.train()

print("\n🎉 Training complete!")

# Save final model
print("\n💾 Saving final model...")
model.save_pretrained("./luganda-gemma-2b-lora-final")
tokenizer.save_pretrained("./luganda-gemma-2b-lora-final")
print("✅ Model saved to ./luganda-gemma-2b-lora-final")

## 🧪 Step 4: Test the Fine-tuned Model

Let's test your new Luganda tutor!

In [ ]:
from peft import PeftModel

print("🔄 Loading fine-tuned model...")

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Load fine-tuned LoRA weights
model = PeftModel.from_pretrained(base_model, "./luganda-gemma-2b-lora-final")
print("✅ Model loaded\n")

# Test prompts
test_prompts = [
    "Teach me a lesson about greetings in Luganda",
    "What does 'Taata' mean?",
    "How do I say 'Thank you' in Luganda?",
]

def generate_response(prompt):
    formatted_prompt = f"""### Instruction:
{prompt}

### Response:
"""
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the response part
    return response.split("### Response:")[-1].strip()

print("🧪 Testing fine-tuned model:\n")
for i, prompt in enumerate(test_prompts, 1):
    print(f"Test {i}: {prompt}")
    print(f"Response: {generate_response(prompt)}")
    print("="*80)
    print()

## 📥 Step 5: Download Your Model

Download the fine-tuned model to use locally:

1. Click the folder icon on the left sidebar
2. Navigate to `luganda-gemma-2b-lora-final/`
3. Download all files (adapter_config.json, adapter_model.safetensors, etc.)
4. Place them in your project: `backend/models/luganda-gemma-2b-lora/`

Or zip and download:

In [ ]:
!zip -r luganda-model.zip ./luganda-gemma-2b-lora-final
print("✅ Model zipped! Download luganda-model.zip from the files panel.")

## 📝 Using Your Model Locally

After downloading, update your `backend/.env`:

```
AI_PROVIDER=local
LOCAL_MODEL_PATH=./models/luganda-gemma-2b-lora
```

Then start your backend:
```bash
cd backend
python main.py
```

Your custom Luganda tutor will now be running! 🎉